<a href="https://colab.research.google.com/github/ailab-nda/ML/blob/main/yolo_sample_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on Object Detection with YOLO (Google Colab)

This is the lab notebook for the lecture “The Evolution of Object Recognition AI.” Using Ultralytics **YOLO26** (released January 2026) with pre-trained models, you will try the following:

| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | Object detection on an image (inference) |
| 2 | Analyzing the results (boxes, classes, confidence) |
| 3 | Effect of parameters (confidence threshold, class filter, model size) |
| 4 | Application 1: instance segmentation |
| 5 | Application 2: pose estimation |
| 6 | Application 3: object tracking in video |
| 7 | Application 4: real-time recognition with your PC or phone camera |
|  | Exercises |

**Before you start:** in the menu, choose “Runtime” → “Change runtime type” and select **T4 GPU** (CPU also works, but the video parts will be slow). Run the cells from top to bottom with Shift + Enter.

## 0. Setup

`ultralytics` is the official YOLO library. `lap` provides the assignment algorithm used for object tracking (Part 6).

In [ ]:
!pip install -q ultralytics lap

In [ ]:
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ASSETS  # sample images bundled with the library (bus.jpg, zidane.jpg)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

ultralytics.checks()  # show versions, GPU availability, etc.
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", "GPU" if DEVICE == 0 else "CPU")

In [ ]:
def show(img_bgr, title=None, size=8):
    # Helper: display an OpenCV image (BGR) with matplotlib
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

BUS = str(ASSETS / "bus.jpg")
ZIDANE = str(ASSETS / "zidane.jpg")

## 1. Object detection on an image (inference)

`YOLO("yolo26n.pt")` loads a model pre-trained on the COCO dataset (80 classes). The weights file (about 5 MB) is downloaded automatically the first time.

The `n` in the model name stands for nano, the smallest size. Sizes grow as `n` < `s` < `m` < `l` < `x`: larger models are more accurate but slower.

In [ ]:
model = YOLO("yolo26n.pt")

results = model(BUS, device=DEVICE, verbose=False)  # inference; returns a list (one element per image)
r = results[0]

show(r.plot(), "YOLO26n detection")  # plot() returns the image (BGR) with boxes and labels drawn

## 2. Analyzing the results

The detections are stored in `r.boxes`. The main attributes are:

| Attribute | Meaning | Shape |
|---|---|---|
| `xyxy` | Top-left (x1, y1) and bottom-right (x2, y2) of each box, in pixels | (N, 4) |
| `xywh` | Box center (x, y), width and height | (N, 4) |
| `xyxyn` | `xyxy` normalized by the image size | (N, 4) |
| `conf` | Confidence (0–1) | (N,) |
| `cls` | Class index | (N,) |

The mapping from class index to name is in `r.names` (a dictionary).

In [ ]:
print("Image size (height, width):", r.orig_shape)
print("Number of detections:", len(r.boxes))
print("Number of classes:", len(r.names), "e.g.", {k: r.names[k] for k in range(5)})
print()
print("xyxy:\n", r.boxes.xyxy)
print("conf:", r.boxes.conf)
print("cls :", r.boxes.cls)

### Put the results in a table (DataFrame)

Converting the tensors to NumPy arrays and then to a pandas table makes them easier to work with. Tensors on the GPU must first be moved to the CPU with `.cpu()`.

In [ ]:
def boxes_to_df(r):
    b = r.boxes
    xyxy = b.xyxy.cpu().numpy()
    df = pd.DataFrame(xyxy, columns=["x1", "y1", "x2", "y2"]).round(1)
    df.insert(0, "class", [r.names[int(c)] for c in b.cls.cpu().numpy()])
    df.insert(1, "conf", b.conf.cpu().numpy().round(3))
    df["width"] = (df.x2 - df.x1).round(1)
    df["height"] = (df.y2 - df.y1).round(1)
    df["area_ratio_%"] = (df.width * df.height / (r.orig_shape[0] * r.orig_shape[1]) * 100).round(2)
    return df

df = boxes_to_df(r)
df

In [ ]:
# Count per class
df["class"].value_counts()

### Draw the boxes yourself

Instead of relying on `plot()`, draw the boxes from the extracted coordinates. The goal is to check what the coordinates mean.

In [ ]:
img = cv2.imread(BUS)
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
for _, row in df.iterrows():
    ax.add_patch(patches.Rectangle((row.x1, row.y1), row.width, row.height,
                                   fill=False, linewidth=2, edgecolor="orange"))
    ax.text(row.x1, row.y1 - 8, f"{row['class']} {row.conf:.2f}", color="black",
            backgroundcolor="orange", fontsize=9)
ax.axis("off")
ax.set_title("Boxes drawn from r.boxes.xyxy")
plt.show()

### Compute the IoU between boxes

Compute the IoU (Intersection over Union) from the lecture for every pair of detected boxes. It is 0 when boxes do not overlap and 1 when they match exactly.

In [ ]:
def iou(a, b):
    # a, b: [x1, y1, x2, y2]
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)

boxes = df[["x1", "y1", "x2", "y2"]].values
n = len(boxes)
iou_mat = np.array([[iou(boxes[i], boxes[j]) for j in range(n)] for i in range(n)])
pd.DataFrame(iou_mat.round(2),
             index=[f"{i}:{c}" for i, c in enumerate(df["class"])],
             columns=[f"{i}:{c}" for i, c in enumerate(df["class"])])

## 3. Effect of parameters

### 3-1. Confidence threshold `conf`

Detections with confidence below `conf` are discarded (default 0.25). Lowering the threshold reduces misses but adds false detections; raising it does the opposite.

In [ ]:
thresholds = [0.05, 0.25, 0.5, 0.75]
fig, axes = plt.subplots(1, len(thresholds), figsize=(20, 7))
for ax, t in zip(axes, thresholds):
    rt = model(BUS, conf=t, device=DEVICE, verbose=False)[0]
    ax.imshow(cv2.cvtColor(rt.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf={t}: {len(rt.boxes)} boxes")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3-2. Detect a specific class and crop it

Passing `classes=[0]` detects only that class (0 = person). Here we crop each detected person out of the image and show them side by side.

In [ ]:
rp = model(BUS, classes=[0], device=DEVICE, verbose=False)[0]
img = cv2.imread(BUS)
crops = [img[int(y1):int(y2), int(x1):int(x2)] for x1, y1, x2, y2 in rp.boxes.xyxy.cpu().numpy()]

fig, axes = plt.subplots(1, len(crops), figsize=(3 * len(crops), 5))
for i, (ax, c) in enumerate(zip(np.atleast_1d(axes), crops)):
    ax.imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    ax.set_title(f"person #{i}  conf={float(rp.boxes.conf[i]):.2f}")
    ax.axis("off")
plt.show()

### 3-3. Comparing model sizes

Compare nano (`n`), small (`s`) and medium (`m`) on the same image. The first inference includes setup work, so we warm up before measuring time.

In [ ]:
rows = []
for name in ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"]:
    m = YOLO(name)
    m(BUS, device=DEVICE, verbose=False)              # warm-up
    rr = m(BUS, device=DEVICE, verbose=False)[0]
    params = sum(p.numel() for p in m.model.parameters()) / 1e6
    rows.append({"model": name,
                 "params (M)": round(params, 1),
                 "inference (ms)": round(rr.speed["inference"], 1),
                 "detections": len(rr.boxes),
                 "mean conf": round(float(rr.boxes.conf.mean()), 3)})
pd.DataFrame(rows)

### 3-4. Try your own images (optional)

Running the cell below shows a file-picker button. Upload your own photos and run detection on them.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        ru = model(fname, device=DEVICE, verbose=False)[0]
        show(ru.plot(), fname)
        display(boxes_to_df(ru))
except ImportError:
    print("This cell only works on Google Colab.")

## 4. Application 1: instance segmentation

Models with `-seg` output a **pixel-level mask** for each object in addition to its box (“instance segmentation” in the lecture slides). The masks are stored in `r.masks`.

In [ ]:
seg_model = YOLO("yolo26n-seg.pt")
rs = seg_model(BUS, device=DEVICE, verbose=False)[0]
show(rs.plot(), "YOLO26n-seg")

print("Mask shape (objects, height, width):", tuple(rs.masks.data.shape))
print("Number of contour points for the first object:", len(rs.masks.xy[0]))

### Mask area vs. box area

Comparing the box area with the mask area shows how much “background” each box contains. Masks are output at the model’s input resolution, so we resize them to the original image size before counting pixels.

In [ ]:
H, W = rs.orig_shape
masks = rs.masks.data.cpu().numpy()
rows = []
for i in range(len(masks)):
    m_full = cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_NEAREST) > 0.5
    x1, y1, x2, y2 = rs.boxes.xyxy[i].cpu().numpy()
    box_area = (x2 - x1) * (y2 - y1)
    rows.append({"class": rs.names[int(rs.boxes.cls[i])],
                 "mask px": int(m_full.sum()),
                 "box px": int(box_area),
                 "mask / box": round(m_full.sum() / box_area, 2)})
pd.DataFrame(rows)

In [ ]:
# Blur the background using only the person masks (a simple "portrait mode")
img = cv2.imread(BUS)
person_mask = np.zeros((H, W), dtype=bool)
for i in range(len(masks)):
    if rs.names[int(rs.boxes.cls[i])] == "person":
        person_mask |= cv2.resize(masks[i], (W, H), interpolation=cv2.INTER_NEAREST) > 0.5

blurred = cv2.GaussianBlur(img, (51, 51), 0)
portrait = np.where(person_mask[..., None], img, blurred)
show(portrait, "Background blurred using person masks")

## 5. Application 2: pose estimation

Models with `-pose` output **17 keypoints (joints)** for each person, in COCO order.

In [ ]:
pose_model = YOLO("yolo26n-pose.pt")
rpo = pose_model(ZIDANE, device=DEVICE, verbose=False)[0]
show(rpo.plot(), "YOLO26n-pose", size=10)

KPT_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
             "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
             "left_wrist", "right_wrist", "left_hip", "right_hip",
             "left_knee", "right_knee", "left_ankle", "right_ankle"]
print("Keypoint shape (people, 17, xy):", tuple(rpo.keypoints.xy.shape))

In [ ]:
# Keypoints of the first person as a table (low-conf points are likely off-screen or occluded)
kp = rpo.keypoints.xy[0].cpu().numpy()
kc = rpo.keypoints.conf[0].cpu().numpy()
pd.DataFrame({"keypoint": KPT_NAMES, "x": kp[:, 0].astype(float).round(1),
              "y": kp[:, 1].astype(float).round(1), "conf": kc.astype(float).round(2)})

### Computing joint angles

Compute the elbow angle from three points: shoulder, elbow and wrist. This is the basic calculation behind applications such as sports form analysis and rehabilitation support.

In [ ]:
def angle(a, b, c):
    # angle abc in degrees, with vertex at point b
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

rows = []
for p in range(len(rpo.keypoints.xy)):
    xy = rpo.keypoints.xy[p].cpu().numpy()
    cf = rpo.keypoints.conf[p].cpu().numpy()
    for side, (s, e, w) in {"left": (5, 7, 9), "right": (6, 8, 10)}.items():
        ok = min(cf[s], cf[e], cf[w]) > 0.5
        rows.append({"person": p, "side": side,
                     "elbow angle (deg)": round(float(angle(xy[s], xy[e], xy[w])), 1) if ok else None,
                     "reliable": ok})
pd.DataFrame(rows)

## 6. Application 3: object tracking in video

Detection treats every frame independently, while **tracking** keeps assigning the same ID to the same object across frames. Here we use the **ByteTrack** algorithm.

So you can try this without your own footage, we first create a **synthetic video** in which a camera pans across the sample image. To use your own video, see 6-4.

In [ ]:
# Create a synthetic video: a 3-second pan (15 fps x 45 frames) across bus.jpg from left to right
img = cv2.imread(BUS)
Hh, Ww = img.shape[:2]
VIDEO = "pan.mp4"
writer = cv2.VideoWriter(VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), 15, (640, 480))
N_FRAMES = 45
for t in range(N_FRAMES):
    x0 = int((Ww - 540) * t / (N_FRAMES - 1))
    frame = cv2.resize(img[300:900, x0:x0 + 540], (640, 480))
    writer.write(frame)
writer.release()
print("Created:", VIDEO)

### 6-1. Run the tracker and record IDs

`model.track(..., persist=True)` runs tracking. With `stream=True` you receive one frame at a time, so even long videos don’t use too much memory.

In [ ]:
track_model = YOLO("yolo26n.pt")
records, annotated = [], []
for f, res in enumerate(track_model.track(VIDEO, stream=True, persist=True, tracker="bytetrack.yaml",
                                          device=DEVICE, verbose=False)):
    annotated.append(res.plot())
    if res.boxes.id is None:
        continue
    for tid, cls, (cx, cy, w, h) in zip(res.boxes.id.int().tolist(), res.boxes.cls.int().tolist(),
                                       res.boxes.xywh.cpu().numpy()):
        records.append({"frame": f, "id": tid, "class": res.names[cls], "cx": cx, "cy": cy})

tracks = pd.DataFrame(records)
print("Frames:", len(annotated), " Tracked IDs:", sorted(tracks["id"].unique().tolist()))
tracks.groupby(["id", "class"]).frame.agg(["min", "max", "count"]).rename(
    columns={"min": "first frame", "max": "last frame", "count": "frames"})

### 6-2. Check a few frames side by side

Check whether each person keeps the same `id` throughout.

In [ ]:
idx = np.linspace(0, len(annotated) - 1, 4).astype(int)
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, i in zip(axes, idx):
    ax.imshow(cv2.cvtColor(annotated[i], cv2.COLOR_BGR2RGB))
    ax.set_title(f"frame {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6-3. Plot trajectories

Connecting the center of each ID over time shows its motion on screen (its trajectory). Because the camera pans right, objects drift to the left on screen.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(cv2.cvtColor(annotated[-1], cv2.COLOR_BGR2RGB), alpha=0.4)
for tid, g in tracks.groupby("id"):
    ax.plot(g.cx, g.cy, "-o", markersize=3, label=f"id {tid} ({g['class'].iloc[0]})")
ax.legend(loc="lower left", fontsize=9)
ax.set_title("Trajectories of tracked objects")
ax.axis("off")
plt.show()

In [ ]:
# Save the tracking result as a video and play it in the notebook (convert to H.264 so browsers can play it)
writer = cv2.VideoWriter("tracked_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 15, annotated[0].shape[1::-1])
for fr in annotated:
    writer.write(fr)
writer.release()
!ffmpeg -y -loglevel error -i tracked_raw.mp4 -vcodec libx264 -pix_fmt yuv420p tracked.mp4

from IPython.display import Video
Video("tracked.mp4", embed=True, width=640)

### 6-4. Try your own video (optional)

Upload a short video (about 10 seconds) taken with your phone; the cell below sets `VIDEO` to its file name. Then re-run the cells from 6-1. Long videos take time, so use a GPU runtime.

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    if up:
        VIDEO = list(up)[0]
        print("Set VIDEO =", VIDEO, "- re-run the cells from 6-1.")
except ImportError:
    print("This cell only works on Google Colab.")

## 7. Application 4: real-time recognition with your camera

Feed live video from your PC’s webcam or your phone’s camera through YOLO.

**How it works:** Colab’s Python runs on a server in the cloud, so it cannot access your camera directly. Instead, JavaScript in your browser turns each camera frame into a JPEG and sends it to the server. Python on the server runs YOLO and sends the result image back to the browser. Repeating this loop gives a near-real-time display.

```
[Browser] camera → JPEG ──send──▶ [Colab server] YOLO detection
     ▲                                            │
     └────────────── result image ◀────────────────┘
```

**Notes**
- The browser will first ask for permission to use the camera. Choose “Allow.”
- Because each frame makes a round trip over the network, expect a few to about 10 fps on a GPU runtime, depending on your connection.
- **Phones:** you can open Colab in your phone’s browser (Chrome / Safari) and run this. To use the rear camera, set `FACING` below to `"environment"`.
- Video is exchanged only between your browser and this Colab session and is not saved (except the photo taken in 7-1).

In [ ]:
# ===== Camera settings =====
FACING = "user"        # "user" = front (PC webcam / selfie camera) / "environment" = phone rear camera
CAPTURE_WIDTH = 640    # width of the images sent to the server (smaller = faster)
JPEG_QUALITY = 0.7     # JPEG quality of the images sent (0–1)

import base64, json, time
from collections import Counter
from IPython.display import display, Javascript, Image as IPyImage

try:
    from google.colab.output import eval_js
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("This part only works on Google Colab (it uses your browser's camera).")

def dataurl_to_bgr(data_url):
    # "data:image/jpeg;base64,...." -> OpenCV image (BGR)
    jpg = base64.b64decode(data_url.split(",", 1)[1])
    return cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)

def bgr_to_dataurl(img_bgr, quality=80):
    ok, buf = cv2.imencode(".jpg", img_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

### 7-1. Start with a single snapshot

Press the “Take photo” button to run detection on that moment’s image. This is also a good way to check that your camera works.

In [ ]:
TAKE_PHOTO_JS = '''
async function takePhoto(facing, width, quality) {
  const div = document.createElement('div');
  const video = document.createElement('video');
  video.setAttribute('playsinline', '');   // keep iPhone Safari from going full screen
  video.muted = true;
  video.style.width = '100%';
  video.style.maxWidth = width + 'px';
  video.style.display = 'block';
  const btn = document.createElement('button');
  btn.textContent = 'Take photo';
  btn.style.fontSize = '18px';
  btn.style.margin = '8px 0';
  div.appendChild(video);
  div.appendChild(btn);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: facing}, audio: false});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

  await new Promise((resolve) => btn.onclick = resolve);

  const scale = width / video.videoWidth;
  const canvas = document.createElement('canvas');
  canvas.width = width;
  canvas.height = Math.round(video.videoHeight * scale);
  canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
  stream.getVideoTracks().forEach(t => t.stop());
  div.remove();
  return canvas.toDataURL('image/jpeg', quality);
}
'''

if IN_COLAB:
    display(Javascript(TAKE_PHOTO_JS))
    data_url = eval_js(f"takePhoto('{FACING}', {CAPTURE_WIDTH}, {JPEG_QUALITY})")
    photo = dataurl_to_bgr(data_url)
    cv2.imwrite("camera_photo.jpg", photo)

    rc = model(photo, device=DEVICE, verbose=False)[0]
    show(rc.plot(), "Camera snapshot")
    display(boxes_to_df(rc))

### Exercises: Real-time object recognition

Please create a program that satisfies the specifications below.

==============================================

Running the cell below shows the detection results in a large view and the live camera feed in a small view at the lower right. The bar at the top shows the processing speed (FPS) and how many objects of each class were detected.

**To stop:** press the “Stop” button or click the result view. It also stops automatically after `MAX_SECONDS` seconds.

Change `RT_MODEL` to `"yolo26n-seg.pt"` (segmentation) or `"yolo26n-pose.pt"` (pose estimation) to try the other tasks in real time.

### Troubleshooting

| Symptom | What to do |
|---|---|
| No camera permission prompt / the view stays black | Click the lock icon next to the address bar, set Camera to “Allow,” and re-run the cell |
| Errors such as `NotReadableError` | Check that no other app (Zoom, Teams, etc.) is using the camera |
| Too slow | Use a GPU runtime, or lower `CAPTURE_WIDTH` to 480 or 320 |
| Phone doesn’t use the rear camera | Set `FACING = "environment"` and re-run from the settings cell |
| The cell doesn’t stop | Press “Stop.” If it doesn’t respond, press ■ (interrupt execution) in the toolbar |